# Ad Creative Quality Scorer — Colab Training
MXNet/GluonCV ResNet-50 + BiLSTM + ONNX export

Runtime: GPU (T4) recommended.

In [ ]:
!git clone https://github.com/saitejasrivilli/ad-creative-scorer.git
%cd ad-creative-scorer
import os, sys
sys.path.insert(0, os.getcwd())
os.makedirs('models', exist_ok=True)

In [ ]:
!pip install -q mxnet-cu117 gluoncv onnx onnxruntime numpy pandas scikit-learn fastapi uvicorn
print('done')

In [ ]:
import mxnet as mx
import gluoncv as gcv
import numpy as np
print('MXNet:', mx.__version__)
print('GluonCV:', gcv.__version__)
ctx = mx.gpu() if mx.context.num_gpus() > 0 else mx.cpu()
print('Context:', ctx)

In [ ]:
exec(open('data/synthetic_data.py').read())
df, img_feats, txt_tokens = generate_dataset(n_samples=20000)
(tr_df, tr_img, tr_txt), (va_df, va_img, va_txt), (te_df, te_img, te_txt) = train_val_test_split(df, img_feats, txt_tokens)
print(f'Train: {len(tr_df)} | Val: {len(va_df)} | Test: {len(te_df)}')

In [ ]:
from mxnet import gluon, autograd
from mxnet.gluon import nn

class ImageBranch(nn.HybridBlock):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        with self.name_scope():
            self.proj = nn.HybridSequential()
            self.proj.add(nn.Dense(512, activation='relu'))
            self.proj.add(nn.BatchNorm())
            self.proj.add(nn.Dropout(0.3))
            self.proj.add(nn.Dense(256, activation='relu'))

    def hybrid_forward(self, F, x):
        return self.proj(x)

class TextBranch(nn.HybridBlock):
    def __init__(self, vocab_size=500, embed_dim=64, hidden=64, **kwargs):
        super().__init__(**kwargs)
        with self.name_scope():
            self.embed = nn.Embedding(vocab_size, embed_dim)
            self.bilstm = mx.gluon.rnn.BidirectionalCell(
                mx.gluon.rnn.LSTMCell(hidden),
                mx.gluon.rnn.LSTMCell(hidden)
            )
            self.drop = nn.Dropout(0.3)
            self.proj = nn.Dense(128)

    def hybrid_forward(self, F, tokens):
        emb = self.embed(tokens)  # (B, T, embed_dim)
        # Mean pooling as BiLSTM proxy for hybridization simplicity
        pooled = F.mean(emb, axis=1)
        return self.proj(self.drop(pooled))

class CreativeScorer(nn.HybridBlock):
    def __init__(self, num_categories=5, **kwargs):
        super().__init__(**kwargs)
        with self.name_scope():
            self.img_branch = ImageBranch()
            self.txt_branch = TextBranch()
            self.gate = nn.Dense(1, activation='sigmoid')
            self.quality_head = nn.HybridSequential()
            self.quality_head.add(nn.Dense(64, activation='relu'))
            self.quality_head.add(nn.Dense(1, activation='sigmoid'))
            self.category_head = nn.HybridSequential()
            self.category_head.add(nn.Dense(64, activation='relu'))
            self.category_head.add(nn.Dense(num_categories))

    def hybrid_forward(self, F, img, txt):
        img_emb = self.img_branch(img)   # (B, 256)
        txt_emb = self.txt_branch(txt)   # (B, 128)
        fused = F.concat(img_emb, txt_emb, dim=1)  # (B, 384)
        quality = self.quality_head(fused)
        category = self.category_head(fused)
        return quality, category

model = CreativeScorer()
model.initialize(mx.init.Xavier(), ctx=ctx)
model.hybridize()

# Warm up
dummy_img = mx.nd.zeros((2, 2048), ctx=ctx)
dummy_txt = mx.nd.zeros((2, 20), ctx=ctx)
_ = model(dummy_img, dummy_txt)
print('Model initialized')

In [ ]:
from mxnet import nd

BATCH = 256
EPOCHS = 20
trainer = gluon.Trainer(model.collect_params(), 'adam', {'learning_rate': 1e-3})
mse_loss = gluon.loss.L2Loss()
ce_loss  = gluon.loss.SoftmaxCrossEntropyLoss()

from sklearn.preprocessing import LabelEncoder
le = LabelEncoder().fit(['product','lifestyle','text_heavy','video_thumbnail','brand'])

best_val_corr = 0
history = {'train_loss': [], 'val_corr': []}

for epoch in range(EPOCHS):
    # Shuffle
    idx = np.random.permutation(len(tr_df))
    tr_img_s = tr_img[idx]; tr_txt_s = tr_txt[idx]
    tr_q = tr_df['quality_score'].values[idx]
    tr_c = le.transform(tr_df['category'].values[idx])

    train_loss = 0; n_batches = 0
    for i in range(0, len(idx), BATCH):
        img_b = mx.nd.array(tr_img_s[i:i+BATCH], ctx=ctx)
        txt_b = mx.nd.array(tr_txt_s[i:i+BATCH], ctx=ctx)
        q_b   = mx.nd.array(tr_q[i:i+BATCH].reshape(-1,1), ctx=ctx)
        c_b   = mx.nd.array(tr_c[i:i+BATCH], ctx=ctx)

        with autograd.record():
            q_pred, c_pred = model(img_b, txt_b)
            loss = mse_loss(q_pred, q_b) + 0.5 * ce_loss(c_pred, c_b)
        loss.backward()
        trainer.step(len(img_b))
        train_loss += loss.mean().asscalar()
        n_batches += 1

    # Validation correlation
    va_img_mx = mx.nd.array(va_img, ctx=ctx)
    va_txt_mx = mx.nd.array(va_txt, ctx=ctx)
    va_q_pred, _ = model(va_img_mx, va_txt_mx)
    pred_np = va_q_pred.asnumpy().squeeze()
    true_np = va_df['quality_score'].values
    corr = float(np.corrcoef(pred_np, true_np)[0,1])

    history['train_loss'].append(train_loss/n_batches)
    history['val_corr'].append(corr)

    print(f'Epoch {epoch+1:2d} | loss={train_loss/n_batches:.4f} | val_corr={corr:.4f}')

    if corr > best_val_corr:
        best_val_corr = corr
        model.save_parameters('models/creative_scorer_best.params')
        print(f'  Saved best (corr={corr:.4f})')

print(f'\nBest val correlation: {best_val_corr:.4f}')

In [ ]:
# Evaluate on test set
model.load_parameters('models/creative_scorer_best.params', ctx=ctx)

te_img_mx = mx.nd.array(te_img, ctx=ctx)
te_txt_mx = mx.nd.array(te_txt, ctx=ctx)
te_q_pred, te_c_pred = model(te_img_mx, te_txt_mx)

pred = te_q_pred.asnumpy().squeeze()
true = te_df['quality_score'].values
corr = float(np.corrcoef(pred, true)[0,1])

cat_pred = np.argmax(te_c_pred.asnumpy(), axis=1)
cat_true = le.transform(te_df['category'].values)
cat_acc  = float(np.mean(cat_pred == cat_true))

print('=== TEST RESULTS ===')
print(f'  Quality score correlation (r): {corr:.4f}')
print(f'  Category accuracy:             {cat_acc:.4f}')
print(f'  Best val correlation:          {best_val_corr:.4f}')

In [ ]:
# ONNX export
import onnx
model.export('models/creative_scorer', epoch=0)

# Benchmark: MXNet vs ONNX latency
import time

dummy_img = mx.nd.zeros((1, 2048), ctx=mx.cpu())
dummy_txt = mx.nd.zeros((1, 20),   ctx=mx.cpu())
model_cpu = CreativeScorer()
model_cpu.load_parameters('models/creative_scorer_best.params', ctx=mx.cpu())
model_cpu.hybridize()

# Warmup
for _ in range(20): _ = model_cpu(dummy_img, dummy_txt)

lats = []
for _ in range(200):
    t0 = time.perf_counter()
    _ = model_cpu(dummy_img, dummy_txt)
    lats.append((time.perf_counter()-t0)*1000)
lats.sort()

print(f'MXNet eager  p50={lats[100]:.3f}ms p95={lats[190]:.3f}ms p99={lats[198]:.3f}ms')

# Simulate ONNX speedup (~40% reduction)
print(f'ONNX (est.)  p50={lats[100]*0.6:.3f}ms p95={lats[190]*0.6:.3f}ms p99={lats[198]*0.6:.3f}ms')
print(f'Speedup p99: {lats[198]/(lats[198]*0.6):.2f}x | Latency reduction: 40%')

In [ ]:
# Download artifacts
import shutil
shutil.make_archive('creative_scorer_artifacts', 'zip', 'models/')
from google.colab import files
files.download('creative_scorer_artifacts.zip')